# XAI on Colab: Grad-CAM and Integrated Gradients

**Before running:** Runtime → Change runtime type → **T4 GPU**.

What this notebook does:
- **Sections 5–8:** Grad-CAM **and** IG on the 860 test tumor images, for the representative seed of each model (about 30–45 min per model). Used for RQ2 and RQ3.
- **Section 9:** Grad-CAM only for the other 8 runs (a few minutes each). Used for RQ4.

Results are written to Google Drive as they go. **If the session disconnects:** run cells 1–4 again, then rerun the cell that was interrupted. Images already done are skipped automatically.

## 1. Connect Google Drive
The trained models are read from Drive, and the results are saved there.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code and data
Clones the repo the first time; pulls the latest code if it's already there.

In [ ]:
import os

REPO = "/content/COMP-6936_Project"
if os.path.exists(REPO):
    %cd {REPO}
    !git pull
else:
    !git clone https://github.com/labib-islam/COMP-6936_Project.git {REPO}
    %cd {REPO}

!git log --oneline -1

## 3. Check the GPU
Should print `True` and `Tesla T4`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 4. Settings
`RUNS` is the Drive folder with the 12 trained runs (each with `best.pt`). `OUT` is where the XAI results go.

The representative seeds come from `results/test_summary.csv`: the seed whose val F1 is closest to the model's mean.

In [ ]:
DRIVE = "/content/drive/MyDrive/COMP6936 - Advanced Machine Learning"
RUNS = f"{DRIVE}/runs"
OUT = f"{DRIVE}/xai"

REP_SEEDS = {"densenet121": 42, "convnext_tiny": 43, "resnet50": 43, "efficientnet_v2_s": 44}
ALL_SEEDS = [42, 43, 44]

os.makedirs(OUT, exist_ok=True)

# check that all 12 checkpoints are on Drive
for model in REP_SEEDS:
    for seed in ALL_SEEDS:
        path = f"{RUNS}/{model}_lr0.0001_seed{seed}/best.pt"
        print("ok     " if os.path.exists(path) else "MISSING", path)

## 5. Quick checks
The score unit tests, then a 3-image run to a temporary folder (not Drive). Both take under a minute.

In [ ]:
!python -m src.xai.scores
!python -u -m src.xai.explain --model densenet121 --seed 42 --runs "{RUNS}" --out /content/xai_test --limit 3

## 6. DenseNet121 (selected model): Grad-CAM + IG

In [ ]:
!python -u -m src.xai.explain --model densenet121 --seed 42 --runs "{RUNS}" --out "{OUT}"

## 7. ConvNeXt-Tiny: Grad-CAM + IG

In [ ]:
!python -u -m src.xai.explain --model convnext_tiny --seed 43 --runs "{RUNS}" --out "{OUT}"

## 8. ResNet50: Grad-CAM + IG

In [ ]:
!python -u -m src.xai.explain --model resnet50 --seed 43 --runs "{RUNS}" --out "{OUT}"

## 9. EfficientNetV2-S: Grad-CAM + IG

In [ ]:
!python -u -m src.xai.explain --model efficientnet_v2_s --seed 44 --runs "{RUNS}" --out "{OUT}"

## 10. Grad-CAM only for the other 8 runs (for RQ4)

In [ ]:
for model, rep_seed in REP_SEEDS.items():
    for seed in ALL_SEEDS:
        if seed != rep_seed:
            !python -u -m src.xai.explain --model {model} --seed {seed} --runs "{RUNS}" --out "{OUT}" --methods gradcam

## 11. Overview of everything on Drive
Mean scores for the predicted class. Chance level: pointing = tumor area share (about 0.02), energy ratio = 1.

In [ ]:
import glob
import pandas as pd

frames = [pd.read_csv(f) for f in glob.glob(f"{OUT}/*/*_scores.csv")]
scores = pd.concat(frames)
scores = scores[scores["target"] == "pred"]
overview = scores.groupby(["model", "seed", "method"]).agg(
    n=("path", "count"), pointing=("pointing_hit", "mean"),
    energy=("energy", "mean"), energy_ratio=("energy_ratio", "mean"))
overview.round(3)

## 12. After everything has run
- Download the `xai` folder from Drive (`COMP6936 - Advanced Machine Learning/xai/`) into the repo as `results/xai/`.
- The `*_scores.csv` files get committed. The `maps/` folders are gitignored (example heatmaps for figures only).